# Neural networks from scratch

**Goal:** understand exactly what happens inside a neural network (forward pass, loss, gradients, backpropagation,
gradient descent) by writing one in plain numpy, checking it, and then rebuilding it in PyTorch.

Read `docs/neural_networks.md` alongside this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import make_moons

from asthunt.models.train_baseline_classifier import train_baseline_classifier

rng = np.random.default_rng(42)
X, y = make_moons(n_samples=600, noise=0.25, random_state=42)       # two interleaved half-circles
X_train, y_train, X_test, y_test = X[:400], y[:400], X[400:], y[400:]
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10); plt.title("two classes that no straight line can separate"); plt.show()

## 1. A straight line fails

In [ ]:
def plot_boundary(predict, title):
    xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    plt.contourf(xx, yy, predict(grid).reshape(xx.shape), levels=20, cmap="coolwarm", alpha=0.5)
    plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.2)
    plt.title(title); plt.show()

logistic = train_baseline_classifier(X_train, y_train)
plot_boundary(lambda g: logistic.predict_proba(g)[:, 1], f"logistic regression: test accuracy {logistic.score(X_test, y_test):.2f}")

## 2. A neural network in numpy
One **hidden layer** of 16 neurons. Each neuron computes `relu(w · x + b)`, with ReLU(z) = max(0, z).
The output neuron combines them: `p = sigmoid(w2 · hidden + b2)`.

Stacking simple bent lines (ReLUs) lets the network draw curved boundaries.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def initialise(n_inputs, n_hidden, rng):
    # Small random weights: if all started equal, every neuron would learn the same thing
    return {
        "W1": rng.normal(0, 1 / np.sqrt(n_inputs), (n_inputs, n_hidden)), "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, 1)),        "b2": np.zeros(1),
    }


def forward(params, X):
    """Run the network; also return the in-between values needed for backpropagation."""
    z1 = X @ params["W1"] + params["b1"]          # (n, hidden): weighted sums
    h = np.maximum(0, z1)                          # ReLU
    z2 = h @ params["W2"] + params["b2"]           # (n, 1)
    p = sigmoid(z2)[:, 0]                          # probabilities
    return p, (X, z1, h)


def loss(p, y):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))   # binary cross-entropy

### Backpropagation: the chain rule, layer by layer
We need how the loss changes when each weight changes (its **gradient**). Work backwards from the output:
1. For sigmoid + cross-entropy, the gradient at the output is simply `p - y` (a neat cancellation; see the doc).
2. Output layer: `dW2 = hᵀ · (p - y) / n`.
3. Pass the error back through W2, then through ReLU (its slope is 1 where z1 > 0, else 0).
4. Hidden layer: `dW1 = Xᵀ · error_hidden / n`.

In [ ]:
def backward(params, p, y, cache):
    X, z1, h = cache
    n = len(y)
    error_out = (p - y)[:, None]                             # (n, 1)
    grads = {"W2": h.T @ error_out / n, "b2": error_out.mean(axis=0)}
    error_hidden = (error_out @ params["W2"].T) * (z1 > 0)    # chain rule through ReLU
    grads["W1"] = X.T @ error_hidden / n
    grads["b1"] = error_hidden.mean(axis=0)
    return grads

### Check the gradients numerically
Never trust hand-written gradients. Nudge one weight by a tiny ε and see how the loss moves:
`slope ≈ (loss(w + ε) - loss(w - ε)) / 2ε`. It must match backpropagation.

In [ ]:
params = initialise(2, 16, rng)
p, cache = forward(params, X_train)
grads = backward(params, p, y_train, cache)

epsilon = 1e-5
for name, index in [("W1", (0, 3)), ("W2", (5, 0)), ("b1", (7,))]:
    original = params[name][index]
    params[name][index] = original + epsilon; loss_plus = loss(forward(params, X_train)[0], y_train)
    params[name][index] = original - epsilon; loss_minus = loss(forward(params, X_train)[0], y_train)
    params[name][index] = original
    numeric = (loss_plus - loss_minus) / (2 * epsilon)
    print(f"{name}{index}: backprop {grads[name][index]: .8f}   numeric {numeric: .8f}")

### Gradient descent
Repeat: forward → loss → backward → move every weight a small step **against** its gradient (downhill).

In [ ]:
params = initialise(2, 16, np.random.default_rng(0))
learning_rate = 0.5
losses = []
for step in range(3000):
    p, cache = forward(params, X_train)
    losses.append(loss(p, y_train))
    grads = backward(params, p, y_train, cache)
    for name in params:
        params[name] -= learning_rate * grads[name]

test_accuracy = ((forward(params, X_test)[0] > 0.5) == y_test).mean()
plt.plot(losses); plt.xlabel("step"); plt.ylabel("training loss"); plt.yscale("log"); plt.show()
plot_boundary(lambda g: forward(params, g)[0], f"numpy network: test accuracy {test_accuracy:.2f}")

## 3. The same network in PyTorch
PyTorch does the backward pass for us (**autograd**) and runs on the GPU. The training loop has the same four steps.

In [ ]:
torch.manual_seed(0)
network = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
optimizer = torch.optim.SGD(network.parameters(), lr=0.5)
loss_function = nn.BCEWithLogitsLoss()                  # sigmoid + cross-entropy in one, numerically safer

X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)
for step in range(3000):
    optimizer.zero_grad()
    loss_value = loss_function(network(X_tensor)[:, 0], y_tensor)   # forward + loss
    loss_value.backward()                                            # backward (autograd)
    optimizer.step()                                                 # gradient descent step

def predict(points):
    with torch.no_grad():                     # we only want answers, not gradients
        return torch.sigmoid(network(torch.tensor(points, dtype=torch.float32))[:, 0]).numpy()

accuracy = ((predict(X_test) > 0.5) == y_test).mean()
plot_boundary(predict, f"PyTorch network: test accuracy {accuracy:.2f}")

## 4. Overfitting: when the network memorises
A big network (256 neurons × 2 layers) trained on only 40 points.

In [ ]:
torch.manual_seed(0)
X_small, y_small = torch.tensor(X_train[:40], dtype=torch.float32), torch.tensor(y_train[:40], dtype=torch.float32)
X_val, y_val = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
big = nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))
optimizer = torch.optim.Adam(big.parameters(), lr=0.01)

train_curve, validation_curve = [], []
for step in range(1500):
    optimizer.zero_grad()
    loss_value = loss_function(big(X_small)[:, 0], y_small)
    loss_value.backward(); optimizer.step()
    with torch.no_grad():
        train_curve.append(loss_value.item())
        validation_curve.append(loss_function(big(X_val)[:, 0], y_val).item())

plt.plot(train_curve, label="training loss"); plt.plot(validation_curve, label="validation loss")
best = int(np.argmin(validation_curve))
plt.axvline(best, ls="--", c="gray", label=f"best validation (step {best}): stop here")
plt.yscale("log"); plt.xlabel("step"); plt.legend(); plt.show()

Training loss keeps falling towards zero, while validation loss turns and climbs: the network is memorising the 40 points,
noise included. **Early stopping** keeps the model from the best validation step. Our CNN training does exactly this.

## Exercises
1. Change the hidden layer to 2 neurons. What does the boundary look like? Why?
2. Set `learning_rate = 5.0` in the numpy loop. What happens to the loss curve?
3. Break the backward pass on purpose (drop `* (z1 > 0)`). Does the gradient check catch it?